<a href="https://colab.research.google.com/github/AMLU-ANNA-JOSHY/Anomaly_Detection/blob/main/Elliptic_Bitcoin_fraud_detection_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fraud Detection using GNN on Elliptic Bitcoin Dataset**



### **Elliptic Bitcoin Dataset**

- It is a labeled, graph-structured cryptocurrency fraud dataset built from the Bitcoin blockchain.
- The dataset captures a time-ordered graph of Bitcoin transactions.
- The Bitcoin transactions are represented  as nodes in a directed graph, and the directed flow of Bitcoin between transactions is represented via edges, and the 49 time steps present model the temporal evolution.
- The available labels are licit, illicit, or unknown.
- Used for applications such as anomaly detection, illicit transaction prediction,  and graph-based fraud detection.

In [ ]:
!pip install kaggle

from google.colab import files
files.upload()   # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d ellipticco/elliptic-data-set
!unzip -o elliptic-data-set.zip

In [20]:
import pandas as pd

features_df = pd.read_csv("elliptic_bitcoin_dataset/elliptic_txs_features.csv", header=None)
edges_df = pd.read_csv("elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv")
classes_df = pd.read_csv("elliptic_bitcoin_dataset/elliptic_txs_classes.csv")

print("features shape:", features_df.shape)
print("edges shape:", edges_df.shape)
print("classes shape:", classes_df.shape)

features shape: (203769, 167)
edges shape: (234355, 2)
classes shape: (203769, 2)


In [ ]:
# inspect features df

print("Features head:")
print(features_df.head())

There are no column headers available, so add them explicitly. The first column is transaction id and the other 166 represent time steps.

In [23]:
cols = ["txId", "time_step"] + [f"f{i}" for i in range(165)]
features_df.columns = cols

In [24]:
# inspect classes df

print("Classes head:")
print(classes_df.head())

print("\nUnique classes:")
print(classes_df["class"].unique())

Classes head:
        txId    class
0  230425980  unknown
1    5530458  unknown
2  232022460  unknown
3  232438397        2
4  230460314  unknown

Unique classes:
['unknown' '2' '1']


- The 3 classes are ['unknown', '2', '1'] => [unknown, licit, illicit(fraud)]

In [25]:
# Merge features with labels

df = features_df.merge(classes_df, on="txId")

In [26]:
# remove unknown rows

df = df[df["class"] != "unknown"]

In [27]:
# convert to binary labels: fraud = 1, licit = 0

df["class"] = df["class"].apply(lambda x: 1 if x == "1" else 0)


In [28]:
# inspect again, previously (203769, 167)

print(df.shape)
print(df["class"].value_counts())

(46564, 168)
class
0    42019
1     4545
Name: count, dtype: int64


In [ ]:
df.head()

In [30]:
# inspect edges df

print("Edges head:")
print(edges_df.head())
print("Number of entries:", len(edges_df))

Edges head:
       txId1      txId2
0  230425980    5530458
1  232022460  232438397
2  230460314  230459870
3  230333930  230595899
4  232013274  232029206
Number of entries: 234355


In [31]:
# remove enteries with txId1/2 removed in previous step

valid_nodes = set(df["txId"])

edges_df = edges_df[
    edges_df["txId1"].isin(valid_nodes) &
    edges_df["txId2"].isin(valid_nodes)
]

In [32]:
# inspect edges df again, previous len: 234355

print("Edges head:")
print(edges_df.head())
print("Number of entries:", len(edges_df))

Edges head:
        txId1      txId2
5   232344069   27553029
8     3881097  232457116
15  232051089  232470704
26  230473487    7089694
33  231182296   14660781
Number of entries: 36624


In [33]:
print(edges_df.columns)

Index(['txId1', 'txId2'], dtype='object')


### **GNN using PyTorch Geometric**

- Graph tensors needed for GNN are:
node feature matrix (x), edge index (edge_index), and label vector (y)

In [ ]:
!pip install torch-geometric

In [35]:
# create a mapping of transaction ids to integers

txid_to_idx = {txid: i for i, txid in enumerate(df["txId"].values)}

In [36]:
# node feature matrix: all features except txId and class label

import torch

feature_cols = [c for c in df.columns if c not in ["txId", "class"]]

x = torch.tensor(df[feature_cols].values, dtype=torch.float)
print(x.shape)

torch.Size([46564, 166])


In [37]:
# label vector

y = torch.tensor(df["class"].values, dtype=torch.long)
print(y.shape)

torch.Size([46564])


In [39]:
# edges to index (src-dest) pairs

src = edges_df["txId1"].map(txid_to_idx).values
dst = edges_df["txId2"].map(txid_to_idx).values

In [40]:
# edge_index of shape [2, no. of edges]

edge_index = torch.tensor([src, dst], dtype=torch.long)
print(edge_index.shape)

torch.Size([2, 36624])


/tmp/ipykernel_668/2622094698.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src, dst], dtype=torch.long)


In [41]:
# PyG data object

from torch_geometric.data import Data

data = Data(x=x, edge_index=edge_index, y=y)
print(data)

Data(x=[46564, 166], edge_index=[2, 36624], y=[46564])


In [42]:
# check if mapping is correct

print("Num nodes:", data.num_nodes)
print("Num edges:", data.num_edges)
print("Num node features:", data.num_node_features)
print("Unique labels:", torch.unique(data.y))

Num nodes: 46564
Num edges: 36624
Num node features: 166
Unique labels: tensor([0, 1])


- **GNN built:** Each transaction is represented as a node vector with 166 features; each bitcoin flow is represented via edges; each transaction(node) has an associated label(fraud/licit)
- **GNN problem:** Classify each node as fraud/licit.

In [44]:
# create train/test masks for data

import numpy as np
from sklearn.model_selection import train_test_split

node_indices = np.arange(data.num_nodes)

train_idx, test_idx = train_test_split(
    node_indices,
    test_size=0.2,
    stratify=data.y.numpy(),
    random_state=42
)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

In [45]:
# attach masks to graph and verify that 80-20 split appears
# and ensure fraud count was stratified

data.train_mask = train_mask
data.test_mask = test_mask

print("Train nodes:", int(data.train_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

print("Train fraud count:", int(data.y[data.train_mask].sum()))
print("Test fraud count:", int(data.y[data.test_mask].sum()))

Train nodes: 37251
Test nodes: 9313
Train fraud count: 3636
Test fraud count: 909


### **Graph Convolutional Network (GCN)**

In [49]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [50]:
# GCN model definition

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 2)  # 2 classes: normal, fraud

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv2(x, edge_index)
        return x

In [ ]:
# model initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

model = GCN(in_channels=data.num_node_features, hidden_channels=64).to(device)

In [51]:
# assign class-weights since there is class imbalance
# give more weightage to the minority fraud class

num_normal = int((data.y[data.train_mask] == 0).sum())
num_fraud = int((data.y[data.train_mask] == 1).sum())

class_weights = torch.tensor(
    [1.0, num_normal / num_fraud],
    dtype=torch.float
).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [52]:
# training function

def train():
    model.train()
    optimizer.zero_grad()

    out = model(data)   # shape: [num_nodes, 2]
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()
    return loss.item()

In [53]:
# evaluation function

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out = model(data)

    preds = out[mask].argmax(dim=1)
    probs = F.softmax(out[mask], dim=1)[:, 1]

    y_true = data.y[mask].cpu().numpy()
    y_pred = preds.cpu().numpy()
    y_prob = probs.cpu().numpy()

    return y_true, y_pred, y_prob

In [54]:
# training

for epoch in range(1, 51):
    loss = train()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 10, Loss: 0.4486
Epoch 20, Loss: 0.3443
Epoch 30, Loss: 0.3042
Epoch 40, Loss: 0.2711
Epoch 50, Loss: 0.2538


In [55]:
# testing

y_true, y_pred, y_prob = evaluate(data.test_mask)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

[[7596  808]
 [ 100  809]]
              precision    recall  f1-score   support

           0     0.9870    0.9039    0.9436      8404
           1     0.5003    0.8900    0.6405       909

    accuracy                         0.9025      9313
   macro avg     0.7437    0.8969    0.7921      9313
weighted avg     0.9395    0.9025    0.9140      9313

ROC-AUC: 0.9618617620924396


In [60]:
# threshold tuning to get best recall to avoid false positives

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
best_recall = 0
best_thresh = 0

for t in thresholds:
    y_pred_thr = (y_prob >= t).astype(int)
    r = recall_score(y_true, y_pred_thr)
    if best_recall < r:
        best_recall = r
        best_thresh = t

print("Best threshold: ", best_thresh)

y_pred_thr = (y_prob >= best_thresh).astype(int)

p = precision_score(y_true, y_pred_thr)
r = recall_score(y_true, y_pred_thr)
f1 = f1_score(y_true, y_pred_thr)
cm = confusion_matrix(y_true, y_pred_thr)

print("\nPrecision:", round(p, 4))
print("Recall   :", round(r, 4))
print("F1       :", round(f1, 4))
print("\nConfusion matrix:")
print(cm)

Best threshold:  0.3

Precision: 0.3432
Recall   : 0.934
F1       : 0.5019

Confusion matrix:
[[6779 1625]
 [  60  849]]
